# 📊 Baseball Lab 9: Chart Room
A table full of numbers is hard to read. A **picture** shows the same thing in one look:
when a player got good, when he had his best season, when he slowed down.

Today you'll draw two kinds of charts, and learn the most important rule of chart making:
**a chart without words is a mystery.**

It takes about **40 minutes**. Good plan: do **Parts 1 and 2** one day, and **Parts 3 and 4** another day.

**How it works** (same as always):
- Click a gray box and press **Shift + Enter** to run it.
- `___` means **you fill it in**. There are only two today.
- 🔎 **Google it** gives you words to search if you get stuck.
- A red error is normal. Read the **last line** of it.
- **Guess first!** Before you run a box, say out loud what you think it will show.

First, load the data with the hidden box (click ▶ and wait about 20 seconds).

In [ ]:
#@title ⚾ Load the MLB data, 1871 to today (click ▶ and wait about 20 seconds)
import datetime
import unicodedata

import pandas as pd
import requests

LAHMAN = "https://raw.githubusercontent.com/daviddalpiaz/pylahman/main/data-raw/"
MLB = "https://statsapi.mlb.com/api/v1/"
STATS = ["G", "AB", "H", "HR", "RBI", "SB", "BB", "SO"]


def plain(text):
    # "Acuña" -> "Acuna": kids type names on an English keyboard
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()


def short(text):
    # "Acuña Jr." -> "acu": the start of a last name, to match players across the two sources
    return plain(text).lower()[:3]


# History: the Lahman Baseball Database (SABR, CC BY-SA 3.0). One row per player, season and team.
batting = pd.read_csv(LAHMAN + "Batting.csv")
people = pd.read_csv(LAHMAN + "People.csv")
teams = pd.read_csv(LAHMAN + "Teams.csv")[["yearID", "teamID", "name"]].rename(columns={"name": "team"})
batting = batting.merge(teams, on=["yearID", "teamID"], how="left")

# A player traded during a season has a row for each team: add them up and keep his last team.
per_season = batting.groupby(["playerID", "yearID"], sort=False)
history = per_season[STATS].sum(min_count=1).join(per_season["team"].last()).reset_index()
people["name"] = people["nameFirst"] + " " + people["nameLast"]
history = history.merge(people[["playerID", "name"]], on="playerID").rename(columns={"yearID": "season"})

# The seasons the history book doesn't have yet come live from MLB (statsapi.mlb.com).
lahman_ids = {
    (int(p.birthYear), int(p.birthMonth), int(p.birthDay), short(p.nameLast)): p.playerID
    for p in people.dropna(subset=["birthYear", "birthMonth", "birthDay"]).itertuples()
}
mlb_names, live = {}, []
try:
    for season in range(int(history["season"].max()) + 1, datetime.date.today().year + 1):
        url = MLB + f"stats?stats=season&group=hitting&season={season}&sportId=1&playerPool=ALL&limit=5000"
        found = requests.get(url, timeout=60).json().get("stats", [])
        hitters = found[0]["splits"] if found else []
        if not hitters:
            continue
        roster = requests.get(MLB + f"sports/1/players?season={season}", timeout=60).json()["people"]
        born = {p["id"]: p.get("birthDate", "") for p in roster}
        for h in hitters:
            p, s = h["player"], h["stat"]
            y, m, d = (born.get(p["id"]) or "0-0-0").split("-")
            player_id = lahman_ids.get((int(y), int(m), int(d), short(p["lastName"])), f"mlb{p['id']}")
            mlb_names[player_id] = p["fullName"]
            live.append({
                "playerID": player_id, "season": season, "team": h.get("team", {}).get("name"), "name": p["fullName"],
                "G": s["gamesPlayed"], "AB": s["atBats"], "H": s["hits"], "HR": s["homeRuns"], "RBI": s["rbi"],
                "SB": s["stolenBases"], "BB": s["baseOnBalls"], "SO": s["strikeOuts"],
            })
except (requests.RequestException, ValueError, KeyError) as problem:
    live = []
    print("MLB's live stats didn't answer, so the newest season is missing today. Everything else works.", problem)

stats = pd.concat([history, pd.DataFrame(live)], ignore_index=True)
# Players active today get the name MLB uses now on their whole career ("Vladimir Guerrero Jr."), without accents.
stats["name"] = stats["playerID"].map(mlb_names).fillna(stats["name"]).map(plain)
# Whole numbers (114, not 114.0); a stat nobody counted back then stays empty: <NA>.
stats[STATS] = stats[STATS].astype("Int64")
stats = stats.sort_values(["playerID", "season"], ignore_index=True)
stats = stats[["name", "season", "team"] + STATS + ["playerID"]]
print("Ready!", len(stats), "rows, seasons", stats["season"].min(), "to", stats["season"].max())

---
## Part 1. A picture from two lists 📊
Here are Aaron Judge's home runs in his last five seasons, as two lists (Lab 4).
**Guess first:** what will the picture look like? Which bar will be the tallest?

In [ ]:
import matplotlib.pyplot as plt

seasons = [2022, 2023, 2024, 2025, 2026]
homers = [62, 37, 58, 53, 18]

plt.bar(seasons, homers)
plt.show()

That's a **bar chart**: one bar for each season, and the taller the bar, the more home runs.

| piece | what it does |
|---|---|
| `import matplotlib.pyplot as plt` | opens Python's toolbox for pictures and gives it the short name `plt` (Lab 2: `import ... as`). You saw it in Lab 1 |
| `plt.bar(seasons, homers)` | draws the bars. The **first** list says where each bar stands, left to right. The **second** list says how tall each bar is |
| `plt.show()` | "I'm done drawing, show the picture" |

The two lists go together like a zipper: the 1st season with the 1st number, the 2nd with the 2nd...
`2022` goes with `62`, `2023` goes with `37`, and so on. That's why both lists must be the **same length**.

### A chart without words is a mystery
Show that picture to someone who wasn't here. Whose numbers are these? What do the bars count: home runs? hits? hot dogs?
Nobody can tell. Three more lines fix it:

In [ ]:
plt.bar(seasons, homers)
plt.title("Aaron Judge: home runs each season")
plt.xlabel("Season")
plt.ylabel("Home runs   (taller = more)")
plt.show()

| line | what it writes |
|---|---|
| `plt.title( )` | the headline at the top: **who** and **what** |
| `plt.xlabel( )` | words under the bottom line. Chart makers call the line that goes **across** the **x** line |
| `plt.ylabel( )` | words along the left side. The line that goes **up** is the **y** line |

(How to remember: **x** goes a**cross**. Say "x is a cross".)

Now anybody can read it: Judge's biggest season here was 2022, with 62 home runs. In 2026 he hit only 18.

### ⚾ Your turn: the lying chart
This box has **Shohei Ohtani's** numbers for the same five seasons. But look at the title: it still says Aaron Judge!
A chart with the wrong words is worse than a chart with no words. Fix the title.

In [ ]:
seasons = [2022, 2023, 2024, 2025, 2026]
homers = [34, 44, 54, 55, 30]   # Shohei Ohtani's home runs

plt.bar(seasons, homers, color="dodgerblue")
plt.title("Aaron Judge: home runs each season")   # fix this line
plt.xlabel("Season")
plt.ylabel("Home runs   (taller = more)")
plt.show()

Did you spot `color="dodgerblue"`? That's a **setting**, like `ascending=False` in Lab 2: it changes how the bars are drawn.
Try `"navy"`, `"red"`, `"orange"` or `"green"` if you like.

---
## Part 2. Straight from the data 🗂️
Typing numbers into lists by hand is slow, and it's easy to make a mistake. The table `stats` already has every season
of every player. Let's give its columns straight to `plt.bar`.

**Guess first:** Aaron Judge has played 11 seasons. Which one will have the tallest bar?

In [ ]:
name = "Aaron Judge"
career = stats[stats["name"] == name]

plt.bar(career["season"], career["HR"])
plt.title(name + ": home runs each season")
plt.xlabel("Season")
plt.ylabel("Home runs   (taller = more)")
plt.show()

| line | what it does |
|---|---|
| `career = stats[stats["name"] == name]` | keep only this player's rows: one row per season (Lab 2) |
| `career["season"]` | the season column: where each bar stands |
| `career["HR"]` | the home run column: how tall each bar is |
| `name + ": home runs each season"` | glues the name to the rest of the title with `+` (Lab 8). Now the title can **never lie**: it always shows the player in `name` |

A column of the table works just like a list here. The tallest bar is 2022: his 62 home runs.
And the tiny bar at the start? That's 2016, his first season: he only played 27 games.

### ⚾ Your turn
Draw **your** favorite player's home runs. Change only the name.
(If the chart comes out **empty**, the name is misspelled. It needs capital letters, exactly like in the data.)

In [ ]:
name = "Aaron Judge"   # change to your favorite player
career = stats[stats["name"] == name]

plt.bar(career["season"], career["HR"])
plt.title(name + ": home runs each season")
plt.xlabel("Season")
plt.ylabel("Home runs   (taller = more)")
plt.show()

### Any stat, with the right words
The table has short codes for the stats: `"HR"`, `"SB"`, `"H"`... Good for typing, bad for a chart that mom should understand.
A **dictionary** (Lab 7) can turn each code into plain words: the code is the key, the words are the value.

One piece is missing in this box: the command that writes words along the **left side** (the line that goes up).
It's 6 letters, and you used it a minute ago.

🔎 **Google it:** `matplotlib label the y axis`

**Guess first:** Shohei Ohtani is famous for home runs. In which season did he steal the most bases?

In [ ]:
words = {
    "HR": "Home runs",
    "SB": "Stolen bases",
    "H": "Hits",
    "BB": "Walks",
    "SO": "Strikeouts",
    "G": "Games",
}

name = "Shohei Ohtani"
stat = "SB"
career = stats[stats["name"] == name]

plt.bar(career["season"], career[stat], color="dodgerblue")
plt.title(name + ": " + words[stat] + " each season")
plt.xlabel("Season")
plt.___(words[stat] + "   (taller = more)")
plt.show()

`words[stat]` looks up the code in the dictionary: `words["SB"]` gives `"Stolen bases"`.
So the title and the side label say "Stolen bases", not "SB".

One bar towers over the rest: **2024**, when Ohtani stole 59 bases (and hit 54 home runs in the same season!).

Change `stat` to `"H"` or `"BB"` and run it again: the bars **and** the words change together.

---
## Part 3. The line chart 📈
Bars are great for comparing sizes. But to see how something **changes over time**, there's a second kind of chart:
a **line chart**. It puts a dot for each season and connects the dots, so your eye can follow the ups and downs.

**Guess first:** Babe Ruth played from 1914 to 1935. What will the line of his home runs look like:
going up all the time? Up and then down?

In [ ]:
name = "Babe Ruth"
career = stats[stats["name"] == name]

plt.plot(career["season"], career["HR"], marker="o")
plt.title(name + ": home runs each season")
plt.xlabel("Season")
plt.ylabel("Home runs   (higher = more)")
plt.show()

| piece | what it does |
|---|---|
| `plt.plot( )` | draws a **line** instead of bars. Same two things inside: what goes across, what goes up |
| `marker="o"` | a setting: put a round dot at every season. Without it you get the line only |

Read the line like a story, left to right:
- **1914 to 1917:** flat at the bottom. Ruth was a **pitcher** then, and pitchers rarely bat!
- **1918 to 1921:** the line shoots up. He became a full-time hitter: 29, then 54, then 59 home runs.
- **1927:** the very top, **60** home runs.
- **after 1930:** the line slides down. He was getting older, and 1935 was his last season.

Did you notice the label says `(higher = more)` now, not `(taller = more)`? Lines aren't tall, dots are high. Words must fit the picture.

### ⚾ Your turn
Draw Hank Aaron's career as a line. The command is missing: it's 4 letters, and it draws a line.

🔎 **Google it:** `matplotlib line chart`

In [ ]:
name = "Hank Aaron"
career = stats[stats["name"] == name]

plt.___(career["season"], career["HR"], marker="o")
plt.title(name + ": home runs each season")
plt.xlabel("Season")
plt.ylabel("Home runs   (higher = more)")
plt.show()

Compare the two shapes. Ruth's line has a giant mountain. Aaron's line never goes above 47 (that was 1971),
but it stays high for almost **20 seasons** in a row. Two very different ways to become a home run king!

---
## Part 4. Two careers side by side 🤝
To really compare two players, put both lines on **one** chart. A `for` loop (Lab 5) draws one line for each name in a list.

**Guess first:** Aaron Judge or Shohei Ohtani: who hit more home runs in 2022? And in 2025?

In [ ]:
for name in ["Aaron Judge", "Shohei Ohtani"]:
    career = stats[stats["name"] == name]
    plt.plot(career["season"], career["HR"], marker="o", label=name)

plt.legend()
plt.title("Home runs each season")
plt.xlabel("Season")
plt.ylabel("Home runs   (higher = more)")
plt.show()

| piece | what it does |
|---|---|
| `for name in [...]:` | do the indented lines once for each player (Lab 5) |
| `label=name` | a setting: stick a name tag on this line |
| `plt.legend()` | draws the little box that says which color is which player. Without it, nobody knows who is who! |

The two indented lines run twice, so two lines get drawn, each in its own color. Everything after the loop
(not indented) happens once.

Now the chart answers the question: in 2022 Judge's dot is far above Ohtani's (62 to 34). In 2025 Ohtani is just on top (55 to 53).
They keep trading places!

### A problem: players from different times
Judge and Ohtani play in the same years, so their lines sit on top of each other. Now try Babe Ruth and Aaron Judge.
**Guess first:** what will be wrong with this chart?

In [ ]:
for name in ["Babe Ruth", "Aaron Judge"]:
    career = stats[stats["name"] == name]
    plt.plot(career["season"], career["HR"], marker="o", label=name)

plt.legend()
plt.title("Home runs each season")
plt.xlabel("Season")
plt.ylabel("Home runs   (higher = more)")
plt.show()

Ruth is far on the left, Judge far on the right, with 80 empty years between them. They aren't **side by side** at all.

The fix: don't count by the year on the calendar. Count by **the season of his career**: his 1st season, his 2nd, his 3rd...
Then both players start at 1. `range( )` from Lab 5 makes those numbers:

In [ ]:
for name in ["Babe Ruth", "Aaron Judge"]:
    career = stats[stats["name"] == name]
    number = range(1, len(career) + 1)
    plt.plot(number, career["HR"], marker="o", label=name)

plt.legend()
plt.title("Home runs, season by season of a career")
plt.xlabel("Season of his career   (1 = his first season)")
plt.ylabel("Home runs   (higher = more)")
plt.show()

The new line: `number = range(1, len(career) + 1)`
- `len(career)` is how many seasons he played (Lab 2). Ruth: 22. Judge: 11.
- `range(1, 23)` counts 1, 2, 3 ... 22. It stops **one before** the second number (Lab 5), so we add `+ 1`.

Now you can really compare. Judge started much faster: 52 home runs in his 2nd season, when Ruth was still a pitcher.
And look at season 7: Judge hit 62, Ruth hit 54. Judge's line just stops at 11: his career isn't over yet!

We also changed the words under the chart, because the line across doesn't show years anymore.
**When the chart changes, the words must change too.**

### ⚾ Make it yours: your own match-up
Pick **two players** and **one stat**. Change only the three lines at the top.
The dictionary `words` from Part 2 writes the right words for you.

In [ ]:
player1 = "Babe Ruth"     # change to any player
player2 = "Aaron Judge"   # change to any player
stat = "HR"               # change to "SB", "H", "BB", "SO" or "G"

for name in [player1, player2]:
    career = stats[stats["name"] == name]
    number = range(1, len(career) + 1)
    plt.plot(number, career[stat], marker="o", label=name)

plt.legend()
plt.title(player1 + " and " + player2 + ": " + words[stat])
plt.xlabel("Season of his career   (1 = his first season)")
plt.ylabel(words[stat] + "   (higher = more)")
plt.show()

Show your chart to someone and ask: **"What does this picture tell you?"**
If they can answer without your help, your labels are doing their job.

If something looks odd:
- **only one line, or none:** a name is misspelled (capital letters matter)
- **a red `KeyError`:** the stat isn't in the `words` dictionary. Use one of the six codes
- **a hole in the line of a player from long ago:** nobody counted strikeouts (`SO`) or stolen bases (`SB`) back then

---
### 🏆 Nice work, Chart Maker!
Today you learned:
- `plt.bar( )` draws a **bar chart**: good for comparing sizes
- `plt.plot( )` draws a **line chart**: good for seeing how something changes over time
- both need two things: what goes **across**, and what goes **up**
- `plt.title( )`, `plt.xlabel( )` and `plt.ylabel( )` put words on the chart. A chart without words is a mystery
- `label=` and `plt.legend( )` tell two lines apart
- when the chart changes, the words must change too

Next time is the last lab, and the best one: **your own investigation**. You pick the question, you find the answer
in the data, and you show the proof.